In [1]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'

from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')  # Colab Secrets에 미리 등록해두면 편함
login(token=hf_token)

In [3]:
import shutil
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
import os
DATA_DIR = '/content/drive/MyDrive/비타민/SNU_CV/snuaichallenge_data'
OUTPUT_DIR = '/content/drive/MyDrive/비타민/SNU_CV/snu_qwen3vl_outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(os.listdir(DATA_DIR))  # train.csv, test.csv, train/, test/ 등 확인

['test.csv', 'sample_submission.csv', 'train.csv', 'test', 'train']


In [10]:
!pip install -q -U transformers accelerate peft bitsandbytes qwen-vl-utils

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 770.3/770.3 kB 52.5 MB/s eta 0:00:00


In [5]:
import os, re, ast, random, hashlib, time
import numpy as np
import pandas as pd
from PIL import Image
from concurrent.futures import ThreadPoolExecutor
from collections import defaultdict
from tqdm.auto import tqdm
import networkx as nx
from sklearn.model_selection import StratifiedGroupKFold

import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForImageTextToText, AutoProcessor
from peft import LoraConfig, get_peft_model
from qwen_vl_utils import process_vision_info

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_NAME = "Qwen/Qwen3-VL-8B-Instruct"
N_FOLDS = 5
MAX_SAMPLES_PER_HASH = 3

train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
train['answer_tuple'] = train['Answer'].apply(lambda x: str(ast.literal_eval(x)))

In [6]:
def compute_md5(filepath):
    with open(filepath, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

def get_all_image_records(df, img_dir):
    records = []
    for _, row in df.iterrows():
        for i in range(1, 5):
            path = f"{img_dir}/{row['Id']}/{row[f'Input_{i}']}"
            records.append({'sample_id': row['Id'], 'path': path})
    return records

print("Train 이미지 해시 계산 중...")
train_records = get_all_image_records(train, f'{DATA_DIR}/train')
paths = [r['path'] for r in train_records]
with ThreadPoolExecutor(max_workers=32) as executor:
    hashes = list(tqdm(executor.map(compute_md5, paths), total=len(paths)))
for r, h in zip(train_records, hashes):
    r['hash'] = h

hash_counter = defaultdict(set)
for r in train_records:
    hash_counter[r['hash']].add(r['sample_id'])

hash_to_samples = {h: s for h, s in hash_counter.items() if len(s) <= MAX_SAMPLES_PER_HASH}
excluded = sum(1 for h, s in hash_counter.items() if len(s) > MAX_SAMPLES_PER_HASH)
print(f"제외된 해시(빈번 재사용) 수: {excluded}")

G = nx.Graph()
G.add_nodes_from(train['Id'].unique())
for h, sample_ids in hash_to_samples.items():
    sample_ids = list(sample_ids)
    for i in range(len(sample_ids)):
        for j in range(i+1, len(sample_ids)):
            G.add_edge(sample_ids[i], sample_ids[j])

components = list(nx.connected_components(G))
print(f"총 {len(components)}개 클러스터, 최대 크기={max(len(c) for c in components)}")

id_to_cluster = {sid: cid for cid, comp in enumerate(components) for sid in comp}
train['cluster_id'] = train['Id'].map(id_to_cluster)

sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
train['fold'] = -1
for fold, (_, val_idx) in enumerate(sgkf.split(train, train['answer_tuple'], groups=train['cluster_id'])):
    train.loc[val_idx, 'fold'] = fold

cluster_fold_counts = train.groupby('cluster_id')['fold'].nunique()
print(f"여러 fold에 걸친 클러스터 수: {(cluster_fold_counts > 1).sum()} (0이어야 정상)")
print(train['fold'].value_counts().sort_index())

Train 이미지 해시 계산 중...


  0%|          | 0/38140 [00:00<?, ?it/s]

제외된 해시(빈번 재사용) 수: 2081
총 8147개 클러스터, 최대 크기=14
여러 fold에 걸친 클러스터 수: 0 (0이어야 정상)
fold
0    1915
1    1905
2    1886
3    1911
4    1918
Name: count, dtype: int64


In [7]:
CLAUSE_PATTERN = re.compile(
    r'\b(then|after which|after that|and finally|finally|next|before that|followed by|first|before|after)\b,?',
    flags=re.IGNORECASE
)

def split_clauses(sentence, max_clauses=4):
    sentence = sentence.strip()
    parts = CLAUSE_PATTERN.split(sentence)
    clauses = []
    current = parts[0].strip(' ,;.')
    for i in range(1, len(parts), 2):
        if current:
            clauses.append(current)
        current = parts[i+1].strip(' ,;.')
    if current:
        clauses.append(current)
    clauses = [c for c in clauses if c]
    if len(clauses) > max_clauses:
        clauses = clauses[:max_clauses-1] + [' '.join(clauses[max_clauses-1:])]
    if not clauses:
        clauses = [sentence]
    return clauses

In [8]:
STRUCTURED_HINT = (
    "This caption describes distinct sequential stages, often marked by words like "
    "'then', 'after which', or 'finally'. Match each stage of the story to its corresponding image."
)
CONTINUOUS_HINT = (
    "This caption describes a single continuous action or a brief moment. "
    "Focus on subtle visual cues such as camera movement direction, object position changes, "
    "hand/body motion, and lighting changes to infer the temporal order."
)

def get_prompt_message(row, image_dir):
    img_files = [row['Input_1'], row['Input_2'], row['Input_3'], row['Input_4']]
    sentence = row['Sentence']
    clause_count = len(split_clauses(sentence))
    hint = STRUCTURED_HINT if clause_count >= 3 else CONTINUOUS_HINT

    content = []
    for i, img_file in enumerate(img_files):
        img_path = os.path.join(image_dir, row['Id'], img_file)
        content.append({"type": "image", "image": img_path})
        content.append({"type": "text", "text": f"\nImage {i+1}\n"})

    user_text = (
        f"Thinking about the sentence: \"{sentence}\"\n"
        f"{hint}\n"
        "Look at the 4 images above labeled Image 1 to Image 4. "
        "Determine the correct chronological order of these images to match the sentence. "
        "Provide the answer ONLY as a Python list of integers. "
        "Example: [1, 2, 3, 4]"
    )
    content.append({"type": "text", "text": user_text})
    return [{"role": "user", "content": content}]


def format_training_sample(row, img_dir):
    messages = get_prompt_message(row, img_dir)
    answer = ast.literal_eval(row['Answer'])
    temporal_order = [0, 0, 0, 0]
    for input_idx, pos in enumerate(answer):
        temporal_order[pos - 1] = input_idx + 1
    target_text = str(temporal_order)
    messages.append({"role": "assistant", "content": [{"type": "text", "text": target_text}]})
    return messages, target_text


def parse_model_output(output_text):
    try:
        start_idx = output_text.find('[')
        end_idx = output_text.rfind(']')
        if start_idx != -1 and end_idx != -1:
            result = ast.literal_eval(output_text[start_idx:end_idx+1])
            if isinstance(result, list) and len(result) == 4 and sorted(result) == [1,2,3,4]:
                submission_answer = [0]*4
                for index, image_num in enumerate(result):
                    submission_answer[image_num - 1] = index + 1
                return submission_answer, True
    except:
        pass
    return [1,2,3,4], False

In [3]:
!pip install -q -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.8 MB/s eta 0:00:00


In [9]:
import shutil
cache_path = os.path.expanduser('~/.cache/huggingface/hub')
if os.path.exists(cache_path):
    shutil.rmtree(cache_path)
    print("HF 캐시 삭제 완료, 재다운로드 필요")

In [13]:
import shutil, os
for p in ['~/.cache/huggingface', '/content/qwen3vl_local']:
    full_path = os.path.expanduser(p)
    if os.path.exists(full_path):
        shutil.rmtree(full_path)
        print(f"삭제: {full_path}")

삭제: /root/.cache/huggingface


In [14]:
# 그 다음 모델 로드 재시도
processor = AutoProcessor.from_pretrained(MODEL_NAME)

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# LoRA target_modules 확인 (모델 구조가 다를 수 있으니 먼저 확인)
attn_module_names = set()
for name, _ in model.named_modules():
    if name.split('.')[-1] in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        attn_module_names.add(name.split('.')[-1])
print("발견된 attention 모듈:", attn_module_names)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=list(attn_module_names) if attn_module_names else ["q_proj", "v_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.47k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

발견된 attention 모듈: {'q_proj', 'o_proj', 'k_proj', 'v_proj'}
trainable params: 15,335,424 || all params: 8,782,459,120 || trainable%: 0.1746


In [15]:
class QwenOrderingDataset(Dataset):
    def __init__(self, df, img_dir, processor):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.processor = processor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        messages, target_text = format_training_sample(row, self.img_dir)

        full_text = self.processor.apply_chat_template(messages, tokenize=False)
        prompt_only = self.processor.apply_chat_template(messages[:1], tokenize=False, add_generation_prompt=True)

        image_inputs, _ = process_vision_info(messages)

        full_enc = self.processor(text=[full_text], images=image_inputs, return_tensors="pt", padding=False)
        prompt_enc = self.processor(text=[prompt_only], images=image_inputs, return_tensors="pt", padding=False)

        input_ids = full_enc['input_ids'][0]
        labels = input_ids.clone()
        prompt_len = prompt_enc['input_ids'].shape[1]
        labels[:prompt_len] = -100

        return {
            'input_ids': input_ids,
            'attention_mask': full_enc['attention_mask'][0],
            'labels': labels,
            'pixel_values': full_enc['pixel_values'],
            'image_grid_thw': full_enc['image_grid_thw'],
            'mm_token_type_ids': full_enc['mm_token_type_ids'][0],
        }


def collate_fn(batch):
    max_len = max(item['input_ids'].shape[0] for item in batch)
    pad_id = processor.tokenizer.pad_token_id

    input_ids, attn_mask, labels, mm_token_type_ids = [], [], [], []
    for item in batch:
        seq_len = item['input_ids'].shape[0]
        pad_len = max_len - seq_len
        input_ids.append(torch.cat([item['input_ids'], torch.full((pad_len,), pad_id, dtype=torch.long)]))
        attn_mask.append(torch.cat([item['attention_mask'], torch.zeros(pad_len, dtype=torch.long)]))
        labels.append(torch.cat([item['labels'], torch.full((pad_len,), -100, dtype=torch.long)]))
        mm_token_type_ids.append(torch.cat([item['mm_token_type_ids'], torch.zeros(pad_len, dtype=torch.long)]))

    pixel_values = torch.cat([item['pixel_values'] for item in batch], dim=0)
    image_grid_thw = torch.cat([item['image_grid_thw'] for item in batch], dim=0)

    return {
        'input_ids': torch.stack(input_ids),
        'attention_mask': torch.stack(attn_mask),
        'labels': torch.stack(labels),
        'pixel_values': pixel_values,
        'image_grid_thw': image_grid_thw,
        'mm_token_type_ids': torch.stack(mm_token_type_ids),
    }

In [17]:
import torch
torch.cuda.empty_cache()

# 모델에 gradient checkpointing 적용
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

def safe_save_adapter(model, path):
    tmp_path = path + '_tmp'
    model.save_pretrained(tmp_path)
    if os.path.exists(path):
        shutil.rmtree(path)
    os.rename(tmp_path, path)
    print(f"저장 완료: {path}")

fold = 0
tr_df = train[train['fold'] != fold].reset_index(drop=True)
va_df = train[train['fold'] == fold].reset_index(drop=True)
print(f"Train: {len(tr_df)}, Val: {len(va_df)}")

tr_ds = QwenOrderingDataset(tr_df, f'{DATA_DIR}/train', processor)
tr_loader = DataLoader(tr_ds, batch_size=1, shuffle=True, collate_fn=collate_fn, num_workers=2)  # A100이면 2~4 시도 가능

import bitsandbytes as bnb
optimizer = bnb.optim.AdamW8bit(filter(lambda p: p.requires_grad, model.parameters()), lr=2e-4)
GRAD_ACCUM_STEPS = 8
EPOCHS = 2
CHECKPOINT_EVERY = 500

model.train()
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    total_loss = 0
    for step, batch in enumerate(tqdm(tr_loader, desc=f"Epoch {epoch}")):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss / GRAD_ACCUM_STEPS
        loss.backward()

        if (step + 1) % GRAD_ACCUM_STEPS == 0:
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            optimizer.zero_grad()

        total_loss += loss.item() * GRAD_ACCUM_STEPS
        if step % 100 == 0:
            print(f" step {step}: loss={loss.item()*GRAD_ACCUM_STEPS:.4f}")

        if (step + 1) % CHECKPOINT_EVERY == 0:
            safe_save_adapter(model, f'{OUTPUT_DIR}/qwen3vl_checkpoint_latest')

    print(f"Epoch {epoch} 평균 loss: {total_loss/len(tr_loader):.4f}")
    safe_save_adapter(model, f'{OUTPUT_DIR}/qwen3vl_epoch{epoch}')

safe_save_adapter(model, f'{OUTPUT_DIR}/qwen3vl_final')

Train: 7620, Val: 1915


Epoch 0:   0%|          | 0/7620 [00:00<?, ?it/s]

[transformers] `use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


KeyboardInterrupt: 

In [24]:
import gc
import torch
from transformers import AutoModelForImageTextToText
from peft import PeftModel

# 기존 model이 있다면 정리
try:
    del model
except NameError:
    pass
try:
    del base_model
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# 1. 순수 베이스 모델 새로 로드
base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_NAME,
    dtype=torch.bfloat16,
    device_map="auto",
)

# 2. 학습된 어댑터를 명시적으로 로드 -> 이제 model이 "학습된" 상태가 됨
model = PeftModel.from_pretrained(base_model, f'{OUTPUT_DIR}/qwen3vl_final')
model.eval()

# 3. 확인 - 0이 아니어야 정상
for name, param in model.named_parameters():
    if 'lora_B' in name:
        print(f"LoRA B 값 확인: {param.abs().mean().item()}")
        break

Loading weights:   0%|          | 0/750 [00:00<?, ?it/s]

LoRA B 값 확인: 0.007349385879933834


In [25]:
from scipy.stats import kendalltau

def compute_metrics(pred, target):
    exact_match = (pred == target).all(axis=1).mean()
    position_acc = (pred == target).mean()
    taus = [kendalltau(p,t)[0] if not np.isnan(kendalltau(p,t)[0]) else 1.0 for p,t in zip(pred,target)]
    return {'exact_match': exact_match, 'position_acc': position_acc, 'kendall_tau': np.mean(taus)}

processor.tokenizer.padding_side = 'left'  # 배치 생성을 위해 필수

BATCH_SIZE = 8

model.eval()
results = []

for start_idx in tqdm(range(0, len(va_df), BATCH_SIZE)):
    batch_rows = va_df.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, f'{DATA_DIR}/train') for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

pred_array = np.array([r['pred'] for r in results]) - 1
target_array = np.array([ast.literal_eval(a) for a in va_df['Answer']]) - 1

overall = compute_metrics(pred_array, target_array)
no_order_mask = va_df['No_ordering'].values
shuffled = compute_metrics(pred_array[~no_order_mask], target_array[~no_order_mask])

print(f"[Qwen3-VL-8B + 조건부 프롬프트, Group-CV val, n={len(va_df)}]")
print(f"  전체 exact={overall['exact_match']:.4f}")
print(f"  셔플됨 exact={shuffled['exact_match']:.4f}")
print(f"  파싱 실패율={1-np.mean([r['parse_ok'] for r in results]):.1%}")

  0%|          | 0/240 [00:00<?, ?it/s]

[Qwen3-VL-8B + 조건부 프롬프트, Group-CV val, n=1915]
  전체 exact=0.5462
  셔플됨 exact=0.4913
  파싱 실패율=0.0%


In [26]:
test = pd.read_csv(f'{DATA_DIR}/test.csv')
print(f"Test samples: {len(test)}")

Test samples: 819


In [29]:
import gc

torch.cuda.empty_cache()
gc.collect()

processor.tokenizer.padding_side = 'left'
BATCH_SIZE = 4
TEST_IMAGE_DIR = f'{DATA_DIR}/test'

# 이미 처리된 결과가 있으면 이어서 (재시작 대비)
if 'test_results' not in dir():
    test_results = []

processed_ids = set(r['Id'] for r in test_results)
remaining_test = test[~test['Id'].isin(processed_ids)].reset_index(drop=True)
print(f"남은 샘플 수: {len(remaining_test)} / {len(test)}")

model.eval()
for start_idx in tqdm(range(0, len(remaining_test), BATCH_SIZE)):
    batch_rows = remaining_test.iloc[start_idx:start_idx + BATCH_SIZE]

    batch_messages = [get_prompt_message(row, TEST_IMAGE_DIR) for _, row in batch_rows.iterrows()]
    batch_texts = [processor.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in batch_messages]

    all_image_inputs = []
    for m in batch_messages:
        imgs, _ = process_vision_info(m)
        all_image_inputs.extend(imgs)

    inputs = processor(
        text=batch_texts, images=all_image_inputs,
        padding=True, return_tensors="pt"
    ).to(model.device)

    try:
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=64, do_sample=False)
    except torch.cuda.OutOfMemoryError:
        print(f"OOM 발생, 이 배치는 배치사이즈 1로 재시도")
        torch.cuda.empty_cache()
        # 배치를 1개씩 쪼개서 재시도
        generated_ids_list = []
        for i in range(len(batch_rows)):
            single_input = {k: v[i:i+1] for k, v in inputs.items()}
            with torch.no_grad():
                gid = model.generate(**single_input, max_new_tokens=64, do_sample=False)
            generated_ids_list.append(gid[0])
        # 길이 맞춰 패딩 후 스택 (간단히 개별 디코딩으로 처리)
        trimmed = []
        for i, gid in enumerate(generated_ids_list):
            inp_len = (inputs['input_ids'][i] != processor.tokenizer.pad_token_id).sum().item()
            trimmed.append(gid[len(inputs['input_ids'][i]):])
        output_texts = [processor.tokenizer.decode(t, skip_special_tokens=True) for t in trimmed]
        for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
            pred, parse_ok = parse_model_output(output_text)
            test_results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})
        continue

    trimmed = [out[len(inp):] for inp, out in zip(inputs.input_ids, generated_ids)]
    output_texts = processor.batch_decode(trimmed, skip_special_tokens=True)

    for (_, row), output_text in zip(batch_rows.iterrows(), output_texts):
        pred, parse_ok = parse_model_output(output_text)
        test_results.append({'Id': row['Id'], 'pred': pred, 'parse_ok': parse_ok})

print(f"완료: {len(test_results)} / {len(test)}")
print(f"파싱 실패율: {1 - np.mean([r['parse_ok'] for r in test_results]):.1%}")

남은 샘플 수: 243 / 819


  0%|          | 0/61 [00:00<?, ?it/s]

완료: 819 / 819
파싱 실패율: 0.0%


In [30]:
print(f"지금까지 처리된 개수: {len(test_results)}")

지금까지 처리된 개수: 819


In [31]:
predictions = [{'Id': r['Id'], 'Answer': str(r['pred'])} for r in test_results]
submission_df = pd.DataFrame(predictions)

submit_path = f'{OUTPUT_DIR}/submission_qwen3vl8b.csv'
submission_df.to_csv(submit_path, index=False)

# 형식 검증
sample_submission = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')
assert list(submission_df.columns) == list(sample_submission.columns), "컬럼명 불일치!"
assert len(submission_df) == len(sample_submission), "행 개수 불일치!"
assert set(submission_df['Id']) == set(sample_submission['Id']), "Id 불일치!"

print(f"저장 완료: {submit_path}")
print("형식 검증 통과 — 제출 가능합니다.")
submission_df.head()

저장 완료: /content/drive/MyDrive/비타민/SNU_CV/snu_qwen3vl_outputs/submission_qwen3vl8b.csv
형식 검증 통과 — 제출 가능합니다.


,Id,Answer
0,WI8o3F,"[1, 4, 2, 3]"
1,Ae7zLm,"[2, 3, 4, 1]"
2,4QSIII,"[1, 4, 2, 3]"
3,k7h4gt,"[4, 2, 1, 3]"
4,ZtTwCx,"[1, 3, 2, 4]"


In [32]:
from collections import Counter

pred_strs = [str(r['pred']) for r in test_results]
print("전체 예측 분포 상위 5개:", Counter(pred_strs).most_common(5))

전체 예측 분포 상위 5개: [('[1, 2, 3, 4]', 132), ('[1, 3, 2, 4]', 43), ('[1, 3, 4, 2]', 42), ('[1, 2, 4, 3]', 38), ('[2, 4, 3, 1]', 37)]
